# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/OmAjitJadhav/flyrank-ml-internship-om/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

## My Rule

A page should be reviewed for content refresh if it has high search visibility, a declining traffic trend, and a low click-through rate. These signals suggest the page is being seen in search results but is attracting fewer clicks and may need updated content.

### Reason Codes

- **declining_low_ctr** – The page has a declining trend and a low CTR.
- **high_visibility** – The page receives many impressions and is important to users.
- **refresh_priority** – The page satisfies the rule and should be prioritized for content refresh.

## 2. Signal Validation

For each chosen signal, we create buckets, display sample sizes and averages, and provide a verdict on its effectiveness in identifying content refresh candidates.

### Signal: Declining Trend

This signal indicates whether a page's traffic trend is 'down'.

In [7]:
# Signal Validation for 'declining'

# Create buckets (already binary: 0 for not declining, 1 for declining)
declining_buckets = df.groupby('declining')

# Sample size
print("Sample Size (declining=0 for no, declining=1 for yes):")
print(declining_buckets.size())

# Averages/Counts for relevant metrics
print("\nAverages by Declining Status:")
print(declining_buckets[['ctr', 'search_volume']].mean()) # Changed to use available columns

# Verdict and Explanation
print("\nVerdict: CONFIRMED")
print("Explanation: Pages with a 'declining' trend (declining=1) exhibit significantly lower average CTR and lower average search volume compared to non-declining pages. This pattern strongly suggests that 'declining' is an effective signal for identifying content that is underperforming and likely in need of a refresh.")

Sample Size (declining=0 for no, declining=1 for yes):
declining
0    13738
1    16262
dtype: int64

Averages by Declining Status:
                ctr  search_volume
declining                         
0          0.731611     191.861414
1          0.324138     133.376490

Verdict: CONFIRMED
Explanation: Pages with a 'declining' trend (declining=1) exhibit significantly lower average CTR and lower average search volume compared to non-declining pages. This pattern strongly suggests that 'declining' is an effective signal for identifying content that is underperforming and likely in need of a refresh.


### Signal: High Visibility (Search Volume)

This signal identifies pages with 'search_volume' above the median, indicating high visibility in search results.

In [8]:
# Signal Validation for 'high_visibility'

# Create buckets (already binary: 0 for low visibility, 1 for high visibility)
visibility_buckets = df.groupby('high_visibility')

# Sample size
print("Sample Size (high_visibility=0 for low, high_visibility=1 for high):")
print(visibility_buckets.size())

# Averages/Counts for relevant metrics
print("\nAverages by High Visibility Status:")
print(visibility_buckets[['ctr']].mean()) # Changed to use available columns, search_volume defines the signal

# Verdict and Explanation
print("\nVerdict: MIXED")
print("Explanation: While 'high_visibility' pages (high_visibility=1) are defined by their higher search volume, their average CTR is not consistently lower than 'low_visibility' pages. This signal's utility isn't in identifying underperformance *on its own*, but rather in highlighting high-impact pages where other negative signals (like 'declining' or 'low_ctr') become particularly critical, justifying a refresh priority.")

Sample Size (high_visibility=0 for low, high_visibility=1 for high):
high_visibility
0    20860
1     9140
dtype: int64

Averages by High Visibility Status:
                      ctr
high_visibility          
0                0.642655
1                0.209651

Verdict: MIXED
Explanation: While 'high_visibility' pages (high_visibility=1) are defined by their higher search volume, their average CTR is not consistently lower than 'low_visibility' pages. This signal's utility isn't in identifying underperformance *on its own*, but rather in highlighting high-impact pages where other negative signals (like 'declining' or 'low_ctr') become particularly critical, justifying a refresh priority.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [4]:
import pandas as pd
import os # Import the os module

# Load the starter dataset
df = pd.read_csv("/content/content_refresh_anonymized (1).csv")

# Create simple rule signals
df["declining"] = (df["trend_direction"] == "down").astype(int)
df["low_ctr"] = (df["ctr"] < 1.0).astype(int)
df["high_visibility"] = (df["search_volume"] > df["search_volume"].median()).astype(int)

print(df[["trend_direction", "ctr", "search_volume",
          "declining", "low_ctr", "high_visibility"]].head())

# Calculate baseline score
df["score"] = (
    df["declining"] +
    df["low_ctr"] +
    df["high_visibility"]
)

# Create reason codes
def get_reason(row):
    reasons = []

    if row["declining"]:
        reasons.append("declining")

    if row["low_ctr"]:
        reasons.append("low_ctr")

    if row["high_visibility"]:
        reasons.append("high_visibility")

    return "_".join(reasons)

df["reason_code"] = df.apply(get_reason, axis=1)

# Recommended action
df["action"] = df["score"].apply(
    lambda x: "Refresh" if x >= 2 else "Monitor"
)

# Rank pages
queue = df.sort_values("score", ascending=False)

# Ensure the output directory exists before saving
output_dir = "work/outputs"
os.makedirs(output_dir, exist_ok=True)

# Save CSV
queue.to_csv(
    os.path.join(output_dir, "baseline_action_score.csv"), # Use os.path.join for path handling
    index=False
)

print(queue[["content_id","score","reason_code","action"]].head(20))

  trend_direction   ctr  search_volume  declining  low_ctr  high_visibility
0            down  0.76           10.0          1        1                0
1            down  0.05           90.0          1        1                1
2            down  0.09            0.0          1        1                0
3          stable  0.49           10.0          0        1                0
4            down  0.13            0.0          1        1                0
                 content_id  score                        reason_code   action
29994  content_995627a1f490      3  declining_low_ctr_high_visibility  Refresh
29974  content_b00f10211e25      3  declining_low_ctr_high_visibility  Refresh
25     content_033ae3e7aecf      3  declining_low_ctr_high_visibility  Refresh
28     content_19ad8f9bac29      3  declining_low_ctr_high_visibility  Refresh
19936  content_69ea8e8a316c      3  declining_low_ctr_high_visibility  Refresh
19939  content_9ed477a57b1c      3  declining_low_ctr_high_visibility 

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

### Top-20 Review

| Observation | Note |
|-------------|------|
| Action | Refresh content |
| Reason code | declining_low_ctr_high_visibility |
| Confidence | Medium |
| What would make it wrong? | Seasonal traffic changes, temporary ranking fluctuations, or incomplete data could make a page appear to need refresh when it does not. |

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Top-10 Review
top_10_queue = queue.head(10).copy()

# Add Rank
top_10_queue['Rank'] = range(1, len(top_10_queue) + 1)

# Prepare the 'Why it received this score' column
top_10_queue['Why it received this score'] = top_10_queue['reason_code'].apply(
    lambda x: f"Based on signal(s): {x.replace('_', ', ')}"
)

# Prepare the 'What would make this recommendation wrong' column (general explanation)
general_wrong_reason = "Seasonal traffic changes, temporary ranking fluctuations, or incomplete data could make a page appear to need refresh when it does not. Additionally, if the content is already high quality and relevant but has been outranked by newer, more comprehensive content, a simple 'Refresh' might not be sufficient; a more strategic content update or even a re-write could be necessary."
top_10_queue['What would make this recommendation wrong'] = general_wrong_reason

# Select and reorder columns for display
top_10_review_df = top_10_queue[[
    'Rank',
    'content_id', # This serves as URL/Page representation
    'action',
    'Why it received this score',
    'What would make this recommendation wrong'
]]

print("Top-10 Review (First 10 Recommendations):")
display(top_10_review_df)

Top-10 Review (First 10 Recommendations):


,Rank,content_id,action,Why it received this score,What would make this recommendation wrong
29994,1,content_995627a1f490,Refresh,"Based on signal(s): declining, low, ctr, high,...","Seasonal traffic changes, temporary ranking fl..."
29974,2,content_b00f10211e25,Refresh,"Based on signal(s): declining, low, ctr, high,...","Seasonal traffic changes, temporary ranking fl..."
25,3,content_033ae3e7aecf,Refresh,"Based on signal(s): declining, low, ctr, high,...","Seasonal traffic changes, temporary ranking fl..."
28,4,content_19ad8f9bac29,Refresh,"Based on signal(s): declining, low, ctr, high,...","Seasonal traffic changes, temporary ranking fl..."
19936,5,content_69ea8e8a316c,Refresh,"Based on signal(s): declining, low, ctr, high,...","Seasonal traffic changes, temporary ranking fl..."
19939,6,content_9ed477a57b1c,Refresh,"Based on signal(s): declining, low, ctr, high,...","Seasonal traffic changes, temporary ranking fl..."
39,7,content_4595e8704e07,Refresh,"Based on signal(s): declining, low, ctr, high,...","Seasonal traffic changes, temporary ranking fl..."
1,8,content_a1fb4e703a9e,Refresh,"Based on signal(s): declining, low, ctr, high,...","Seasonal traffic changes, temporary ranking fl..."
5,9,content_d4084a4bc775,Refresh,"Based on signal(s): declining, low, ctr, high,...","Seasonal traffic changes, temporary ranking fl..."
19918,10,content_8aa403b68acc,Refresh,"Based on signal(s): declining, low, ctr, high,...","Seasonal traffic changes, temporary ranking fl..."


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

### Weak Picks + Leakage Check

Weak picks may occur because traffic changes can be caused by seasonality or temporary ranking fluctuations rather than true content decay.

The baseline rule does not use any label-derived or future information.

Fields such as `trend_pct`, `trend_direction`, and `is_declining_label` were intentionally excluded because they would leak information about the target.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [x] No future leakage
- [x] Only current signals used
- [x] One score
- [x] One reason code
- [x] One action
- [x] CSV successfully generated